# bg

> Synchronous access to background terminal sessions, with paged reads

`ptymini.bg` provides the [bgterm](https://github.com/AnswerDotAI/bgterm) API for synchronous callers, including scripts and kernel tools. Start a terminal once, then use its session ID to send input and read new output in later calls. Reads remember their position and return only unread output. You do not need an asyncio event loop.

In [ ]:
from ptymini.bg import *
from nbdev.showdoc import show_doc


In [ ]:
import sys, time
from fastcore.test import test_eq

## PollResult

`read`, `poll` and `write_stdin` return a `PollResult` containing unread bytes, decoded text and the child process's status. Waits block the calling thread in the Rust core and release the GIL.


In [ ]:
show_doc(PollResult)


## Sessions

`start_bgterm` returns a session ID for subsequent calls. By default, string commands run through `/bin/sh -c`. The child environment always has `TERM=dumb`. A negative exit code identifies the signal that terminated the child, as in `subprocess`.

The wait parameters follow `fastmux.bg`:

- `wait_ms` limits the wait for new output or an `until` match.
- `until` is a regular expression to match against unread text.
- `settle_ms` adds a wait for output to stop.

The total wait cannot exceed `wait_ms + settle_ms`. New output or process exit wakes the waiting call. There is no polling interval parameter.

Each read advances the session's cursor. `until` cannot match text returned by an earlier read. Choose a pattern specific to the output you expect, such as a new prompt.

In [ ]:
show_doc(start_bgterm)


In [ ]:
show_doc(write_stdin)

In [ ]:
show_doc(poll)

In [ ]:
show_doc(read)

In [ ]:
show_doc(wait)

In [ ]:
show_doc(terminate)

In [ ]:
show_doc(kill)

In [ ]:
show_doc(close_bgterm)

In [ ]:
show_doc(list_sessions)

`Session` wraps a session ID for use in a `with` block.

In [ ]:
show_doc(Session)


Start a child process and wait for its banner. Send input with `write_stdin`, using `until` to wait for the reply in the same call.

In [ ]:
sid = start_bgterm([sys.executable, '-u', '-c',
    "print('ready', flush=True)\nimport sys\nfor line in sys.stdin: print(f'ACK:{line.strip()}', flush=True)"])
first = poll(sid, 5000)
assert 'ready' in first.text
r = write_stdin(sid, 'hello\n', 2000, until=r'ACK:hello')
assert 'ACK:hello' in r.text
assert sid in list_sessions()
r.text

Use `settle_ms` when a reply arrives in several bursts without a known terminator. The call collects output until it has been quiet for this duration, subject to the total wait limit. Process exit ends the wait immediately.

In [ ]:
sid2 = start_bgterm([sys.executable, '-u', '-c',
    "import time\nprint('part one', flush=True)\ntime.sleep(0.2)\nprint('part two', flush=True)"])
r = poll(sid2, 5000, settle_ms=500)
assert 'part one' in r.text and 'part two' in r.text
close_bgterm(sid2)
r.text

`PollResult` distinguishes output left to read from output lost to buffer overflow:

- `remaining_bytes` counts unread bytes still in the buffer.
- `dropped_bytes` counts unread bytes discarded before this read.
- `truncated` is true if either count is nonzero.

In [ ]:
sid2 = start_bgterm([sys.executable, '-u', '-c', "import sys; sys.stdout.write('x'*4096); sys.stdout.flush()"], max_buffer_bytes=512)
time.sleep(0.3)
r = read(sid2, 256)
test_eq((r.bytes_returned, r.dropped_bytes, r.truncated), (256, 4096 - 512, True))
r2 = read(sid2, None)
test_eq((r2.bytes_returned, r2.remaining_bytes), (256, 0))
close_bgterm(sid2)
close_bgterm(sid)
with Session.start([sys.executable, '-c', "print('bye'); raise SystemExit(3)"]) as sess:
    assert 'bye' in sess.poll(3000).text
    test_eq(sess.wait(3000), 3)
list_sessions()